In [15]:
#paden zetten
from pathlib import Path
import os

def vind_project_root(marker_map: str = "data") -> Path:
    """
    Zoekt vanaf de huidige working directory (waar je notebook draait)
    omhoog tot we een map vinden die 'data' bevat.
    """
    p = Path.cwd().resolve()
    for _ in range(15):  # max 15 niveaus omhoog zoeken
        if (p / marker_map).exists():
            return p
        p = p.parent
    raise FileNotFoundError("Project-root niet gevonden. Kan map 'data' niet vinden in bovenliggende mappen.")

# Bepaal project-root en bouw paden naar data
PROJECT_ROOT = vind_project_root()

DATA_REQUESTS = PROJECT_ROOT / "data" / "raw" / "requests"
DATA_RESPONSES = PROJECT_ROOT / "data" / "raw" / "responses"
DATA_EXCEL = PROJECT_ROOT / "data" / "raw" / "excel"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_REQUESTS:", DATA_REQUESTS, "| bestaat?", DATA_REQUESTS.exists())
print("DATA_RESPONSES:", DATA_RESPONSES, "| bestaat?", DATA_RESPONSES.exists())
print("DATA_EXCEL:", DATA_EXCEL, "| bestaat?", DATA_EXCEL.exists())

assert DATA_REQUESTS.exists(), "requests-map niet gevonden. Pad/working directory klopt niet."
assert DATA_RESPONSES.exists(), "responses-map niet gevonden. Pad/working directory klopt niet."

PROJECT_ROOT: C:\Users\aasus\PycharmProjects\DataScientistGroepswerk
DATA_REQUESTS: C:\Users\aasus\PycharmProjects\DataScientistGroepswerk\data\raw\requests | bestaat? True
DATA_RESPONSES: C:\Users\aasus\PycharmProjects\DataScientistGroepswerk\data\raw\responses | bestaat? True
DATA_EXCEL: C:\Users\aasus\PycharmProjects\DataScientistGroepswerk\data\raw\excel | bestaat? True


In [16]:
import json
import pandas as pd


In [17]:
# check: Hoeveel bestanden?
request_files = list(DATA_REQUESTS.rglob("*.json"))
response_files = list(DATA_RESPONSES.rglob("*.txt"))

print("Aantal request JSON bestanden:", len(request_files))
print("Aantal response TXT bestanden:", len(response_files))

print("\nVoorbeeld requests:")
for fp in request_files[:5]:
    print(" -", fp)

print("\nVoorbeeld responses:")
for fp in response_files[:5]:
    print(" -", fp)


Aantal request JSON bestanden: 21707
Aantal response TXT bestanden: 21706

Voorbeeld requests:
 - C:\Users\aasus\PycharmProjects\DataScientistGroepswerk\data\raw\requests\0521_300-20220617\0521_300-20220617-055733-2-0.json
 - C:\Users\aasus\PycharmProjects\DataScientistGroepswerk\data\raw\requests\0521_300-20220617\0521_300-20220617-085002-2-0.json
 - C:\Users\aasus\PycharmProjects\DataScientistGroepswerk\data\raw\requests\0521_300-20220617\0521_300-20220617-092416-2-0.json
 - C:\Users\aasus\PycharmProjects\DataScientistGroepswerk\data\raw\requests\0521_300-20220617\0521_300-20220617-124013-2-0.json
 - C:\Users\aasus\PycharmProjects\DataScientistGroepswerk\data\raw\requests\0521_300-20220617\0521_300-20220617-124222-1-0.json

Voorbeeld responses:
 - C:\Users\aasus\PycharmProjects\DataScientistGroepswerk\data\raw\responses\0521_300-20220617\0521_300-20220617-055733-2-0.txt
 - C:\Users\aasus\PycharmProjects\DataScientistGroepswerk\data\raw\responses\0521_300-20220617\0521_300-20220617-08

In [20]:
# Functie: route_id en datum uit foldernaam halen
from typing import Optional, Tuple

def splits_route_en_datum(foldernaam: str) -> Tuple[Optional[str], Optional[str]]:
    """
    Probeert foldernaam te splitten naar (route_id, date).

    Verwacht formaat:
        '0521_300-20220617'
    waarbij:
        route_id = '0521_300'
        date     = '20220617' (exact 8 cijfers)

    Waarom doen we dit robuust?
    - Soms staan er bestanden in een mapnaam die niet aan dit patroon voldoet.
    - In plaats van crashen (IndexError), willen we die bestanden overslaan.
    """
    parts = foldernaam.split("-")

    if len(parts) < 2:
        return None, None

    route_id = parts[0].strip()
    date = parts[1].strip()

    if len(date) != 8 or not date.isdigit():
        return None, None

    return route_id, date


In [21]:
import json
import pandas as pd
from pathlib import Path

def laad_alle_tasks_uit_requests(data_requests: Path) -> pd.DataFrame:
    """
    Leest alle request JSON bestanden in en maakt een DataFrame met tasks.

    We gebruiken uit de data:
    - task_id: identiteit van de locatie (nodig om locaties te volgen)
    - latitude/longitude: om te checken of task_id naar een vaste fysieke plek verwijst
    - route_id en date: om terugkeer over tijd te kunnen analyseren

    Bestanden waarvan de mapnaam niet het formaat 'routeId-YYYYMMDD' heeft,
    worden overgeslagen (omdat we datum/route dan niet betrouwbaar kunnen afleiden).
    """
    rows = []
    json_files = list(data_requests.rglob("*.json"))

    if len(json_files) == 0:
        raise FileNotFoundError(f"Geen JSON files gevonden in {data_requests}. Pad klopt waarschijnlijk niet.")

    skipped = 0

    for fp in json_files:
        obj = json.loads(fp.read_text(encoding="utf-8"))
        folder = fp.parent.name  # bv. 0521_300-20220617

        route_id, date = splits_route_en_datum(folder)

        if route_id is None or date is None:
            skipped += 1
            continue

        for task in obj.get("tasks", []):
            rows.append({
                "route_id": route_id,
                "date": date,
                "task_id": str(task["id"]),
                "latitude": task["address"]["latitude"],
                "longitude": task["address"]["longitude"],
                "request_file": fp.name,
                "request_path": str(fp),
                "configurationName": obj.get("configurationName")
            })

    df = pd.DataFrame(rows)

    print(f"✅ JSON bestanden gevonden: {len(json_files)} | overgeslagen: {skipped}")
    print("✅ df_tasks rijen:", len(df))
    print("✅ df_tasks kolommen:", df.columns.tolist())

    if df.empty:
        raise ValueError("df_tasks is leeg. Dan worden alle bestanden overgeslagen of parsing klopt niet.")

    return df


In [22]:
df_tasks = laad_alle_tasks_uit_requests(DATA_REQUESTS)

print("✅ df_tasks aangemaakt!")
print("Aantal rijen (task-voorkomens):", len(df_tasks))
print("Kolommen:", df_tasks.columns.tolist())
df_tasks.head()


✅ JSON bestanden gevonden: 21707 | overgeslagen: 1
✅ df_tasks rijen: 2576770
✅ df_tasks kolommen: ['route_id', 'date', 'task_id', 'latitude', 'longitude', 'request_file', 'request_path', 'configurationName']
✅ df_tasks aangemaakt!
Aantal rijen (task-voorkomens): 2576770
Kolommen: ['route_id', 'date', 'task_id', 'latitude', 'longitude', 'request_file', 'request_path', 'configurationName']


,route_id,date,task_id,latitude,longitude,request_file,request_path,configurationName
0,0521_300,20220617,395,0.565826,0.221868,0521_300-20220617-055733-2-0.json,C:\Users\aasus\PycharmProjects\DataScientistGr...,CreateSequence
1,0521_300,20220617,394,0.565826,0.221868,0521_300-20220617-055733-2-0.json,C:\Users\aasus\PycharmProjects\DataScientistGr...,CreateSequence
2,0521_300,20220617,385,0.565826,0.221868,0521_300-20220617-085002-2-0.json,C:\Users\aasus\PycharmProjects\DataScientistGr...,CreateSequence
3,0521_300,20220617,384,0.565826,0.221868,0521_300-20220617-085002-2-0.json,C:\Users\aasus\PycharmProjects\DataScientistGr...,CreateSequence
4,0521_300,20220617,390,0.565826,0.221868,0521_300-20220617-092416-2-0.json,C:\Users\aasus\PycharmProjects\DataScientistGr...,CreateSequence


Wat is een locatie?:
-In deze dataset definiëren we een locatie als een task_id.

Waarom?
Omdat routes in responses enkel uit task_id’s bestaan.
Daardoor kunnen we alleen “toegevoegd/verwijderd” objectief bepalen via task_id sets.

In [23]:
def id_betrouwbaarheid_coordinaten(df_tasks: pd.DataFrame) -> pd.DataFrame:
    """
    Controleert of we task_id's kunnen vertrouwen als 'locatie-ID'.

    Idee:
    - Als task_id echt één fysieke locatie representeert, dan blijven lat/lon (bijna) constant.
    - We meten spreiding per task_id via min/max latitude en longitude.

    Output:
    - n: aantal keer dat task_id voorkomt
    - lat_range / lon_range: spreiding (hoe groter -> hoe verdachter)
    """
    s = (df_tasks.groupby("task_id")
         .agg(
             n=("task_id", "count"),
             lat_min=("latitude", "min"),
             lat_max=("latitude", "max"),
             lon_min=("longitude", "min"),
             lon_max=("longitude", "max"),
         )
         .reset_index())

    s["lat_range"] = s["lat_max"] - s["lat_min"]
    s["lon_range"] = s["lon_max"] - s["lon_min"]
    return s


stability = id_betrouwbaarheid_coordinaten(df_tasks)

# Top 20 meest verdachte IDs (grootste spreiding)
stability.sort_values(["lat_range", "lon_range"], ascending=False).head(20)

,task_id,n,lat_min,lat_max,lon_min,lon_max,lat_range,lon_range
112727,34744,15,0.037467,0.615519,0.149549,0.803797,0.578052,0.654248
162605,79320,13,0.482250,1.000000,0.164677,0.414549,0.517750,0.249872
162172,78931,23,0.516091,1.000000,0.206096,0.414549,0.483909,0.208454
68538,18989,27,0.500310,0.918686,0.148066,0.348018,0.418376,0.199952
90035,23883,16,0.512236,0.918686,0.147955,0.438982,0.406450,0.291027
81391,22463,29,0.485454,0.877941,0.228212,0.441400,0.392487,0.213188
51738,169401,10,0.221496,0.610664,0.226997,0.546926,0.389168,0.319929
167476,83733,38,0.300086,0.654876,0.172236,0.362367,0.354791,0.190132
52253,169910,6,0.221496,0.539141,0.243737,0.546926,0.317645,0.303189
3514,105546,7,0.479635,0.791204,0.177281,0.546679,0.311570,0.369398


In [24]:
#terugkeer over dagen(komt een locatie later terug?)
def terugkeer_analyse(df_tasks: pd.DataFrame) -> pd.DataFrame:
    """
    Analyseert of task_id's one-off of terugkerend zijn.

    We berekenen per task_id:
    - first_seen: eerste dag dat task_id voorkomt
    - last_seen: laatste dag dat task_id voorkomt
    - n_days: aantal unieke dagen waarop task_id voorkomt

    Interpretatie:
    - n_days == 1: one-off (komt niet terug in dataset)
    - n_days >= 2: terugkerend (komt later opnieuw voor)
    """
    t = (df_tasks.groupby("task_id")["date"]
         .agg(first_seen="min", last_seen="max", n_days="nunique")
         .reset_index())
    return t


timeline = terugkeer_analyse(df_tasks)

print("% one-off (n_days==1):", (timeline["n_days"] == 1).mean())
print("% terugkerend (n_days>=2):", (timeline["n_days"] >= 2).mean())

timeline.sort_values("n_days", ascending=False).head(20)


% one-off (n_days==1): 0.436582568171869
% terugkerend (n_days>=2): 0.563417431828131


,task_id,first_seen,last_seen,n_days
0,0,20220530,20220622,20
114518,36,20220530,20220622,20
109827,34,20220530,20220622,20
108768,33,20220530,20220622,20
107719,32,20220530,20220622,20
106632,31,20220530,20220622,20
105524,30,20220530,20220622,20
105523,3,20220530,20220622,20
103125,29,20220530,20220622,20
98768,28,20220530,20220622,20


In [25]:
#responses inlezen
def laad_alle_routes_uit_responses(data_responses: Path) -> pd.DataFrame:
    """
    Leest alle response TXT bestanden in en zet ze om naar een tabel.

    Waarom?
    - Om te bepalen welke task_id's effectief in een route zitten.
    - Nodig om 'added/removed' tussen twee routes te berekenen.

    Output (1 rij per task in een response-route):
    - route_id, date
    - response_file
    - sequence_index (positie in route)
    - task_id
    """
    rows = []
    txt_files = list(data_responses.rglob("*.txt"))

    if len(txt_files) == 0:
        raise FileNotFoundError(f"Geen TXT files gevonden in {data_responses}. Pad klopt waarschijnlijk niet.")

    for fp in txt_files:
        folder = fp.parent.name  # bv. 0521_300-20220617
        route_id, date = splits_route_en_datum(folder)

        # als mapnaam niet matcht, overslaan
        if route_id is None or date is None:
            continue

        lines = fp.read_text(encoding="utf-8").splitlines()
        task_ids = [ln.strip() for ln in lines if ln.strip() != ""]

        for i, tid in enumerate(task_ids):
            rows.append({
                "route_id": route_id,
                "date": date,
                "response_file": fp.name,
                "response_path": str(fp),
                "sequence_index": i,
                "task_id": str(tid)
            })

    return pd.DataFrame(rows)


df_routes = laad_alle_routes_uit_responses(DATA_RESPONSES)

print("Rijen (task in routes):", len(df_routes))
print("Unieke response files:", df_routes["response_file"].nunique())
df_routes.head()


Rijen (task in routes): 2572033
Unieke response files: 21704


,route_id,date,response_file,response_path,sequence_index,task_id
0,0521_300,20220617,0521_300-20220617-055733-2-0.txt,C:\Users\aasus\PycharmProjects\DataScientistGr...,0,394
1,0521_300,20220617,0521_300-20220617-055733-2-0.txt,C:\Users\aasus\PycharmProjects\DataScientistGr...,1,395
2,0521_300,20220617,0521_300-20220617-085002-2-0.txt,C:\Users\aasus\PycharmProjects\DataScientistGr...,0,384
3,0521_300,20220617,0521_300-20220617-085002-2-0.txt,C:\Users\aasus\PycharmProjects\DataScientistGr...,1,385
4,0521_300,20220617,0521_300-20220617-092416-2-0.txt,C:\Users\aasus\PycharmProjects\DataScientistGr...,0,388


In [27]:
#toevoegen/ verwijderen tussen routes
def added_removed_tussen_twee_responses(df_routes: pd.DataFrame, route_id: str, date: str,
                                       response_a: str, response_b: str) -> dict:
    """
    Berekent welke task_id's toegevoegd/verwijderd zijn tussen twee response-files.
    """
    a = df_routes[(df_routes["route_id"] == route_id) & (df_routes["date"] == date) & (df_routes["response_file"] == response_a)]
    b = df_routes[(df_routes["route_id"] == route_id) & (df_routes["date"] == date) & (df_routes["response_file"] == response_b)]

    set_a = set(a["task_id"])
    set_b = set(b["task_id"])

    removed = sorted(list(set_a - set_b))
    added = sorted(list(set_b - set_a))

    return {
        "route_id": route_id,
        "date": date,
        "response_a": response_a,
        "response_b": response_b,
        "n_removed": len(removed),
        "n_added": len(added),
        "removed_ids": removed,
        "added_ids": added
    }


In [28]:
# Zoek een route_id+date met minstens 2 response-files
counts = (df_routes.groupby(["route_id", "date"])["response_file"]
          .nunique()
          .reset_index(name="n_responses")
          .sort_values("n_responses", ascending=False))

counts.head(10)


,route_id,date,n_responses
1648,0521_658,20220609,28
3324,0521_O27,20220531,23
2659,0521_879,20220620,23
1204,0521_626,20220609,21
575,0521_364,20220602,21
3466,0521_O42,20220610,21
3459,0521_O42,20220601,21
1647,0521_658,20220608,20
1203,0521_626,20220608,20
2471,0521_867,20220607,19


In [29]:
route_id = counts.iloc[0]["route_id"]
date = counts.iloc[0]["date"]

subset = df_routes[(df_routes["route_id"] == route_id) & (df_routes["date"] == date)]
response_files = sorted(subset["response_file"].unique())

print("Gekozen route_id:", route_id)
print("Gekozen date:", date)
print("Aantal responses die dag:", len(response_files))
print("Eerste 5 response files:", response_files[:5])


Gekozen route_id: 0521_658
Gekozen date: 20220609
Aantal responses die dag: 28
Eerste 5 response files: ['0521_658-20220609-084950-179-0.txt', '0521_658-20220609-085128-177-0.txt', '0521_658-20220609-085235-177-0.txt', '0521_658-20220609-085313-177-0.txt', '0521_658-20220609-085428-177-0.txt']


In [30]:
diff = added_removed_tussen_twee_responses(df_routes, route_id, date, response_files[0], response_files[1])
diff


{'route_id': '0521_658',
 'date': '20220609',
 'response_a': '0521_658-20220609-084950-179-0.txt',
 'response_b': '0521_658-20220609-085128-177-0.txt',
 'n_removed': 2,
 'n_added': 0,
 'removed_ids': ['55493', '55495'],
 'added_ids': []}

In [31]:
def removed_komt_later_terug(diff_dict: dict, timeline: pd.DataFrame) -> pd.DataFrame:
    """
    Checkt voor alle removed_ids of ze later dan 'date' nog voorkomen.
    We gebruiken last_seen uit timeline als snelle test.

    Output:
    - task_id
    - last_seen
    - komt_later_terug (True/False)
    """
    current_date = diff_dict["date"]
    t = timeline.set_index("task_id")

    rows = []
    for tid in diff_dict["removed_ids"]:
        if tid in t.index:
            last_seen = str(t.loc[tid, "last_seen"])
            rows.append({
                "task_id": tid,
                "last_seen": last_seen,
                "komt_later_terug": last_seen > str(current_date)
            })
        else:
            rows.append({
                "task_id": tid,
                "last_seen": None,
                "komt_later_terug": False
            })

    return pd.DataFrame(rows)


removed_check = removed_komt_later_terug(diff, timeline)
removed_check.head(30)


,task_id,last_seen,komt_later_terug
0,55493,20220622,True
1,55495,20220622,True
